<a href="https://colab.research.google.com/github/ARTiwary/Ayush-Flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane:** Content click/traffic decline — ranking which (client, content) items are trending down in organic clicks within a month, using `fact_content_daily_performance`.

**⚠️ Before running:** this notebook was drafted without live access to the Hugging Face warehouse (no network access to `huggingface.co` from where it was written). Column names for `fact_content_daily_performance` are my best inference from `SKILL.md` plus the confirmed `gsc_avg_position` name — **run the schema-discovery cell below first** and adjust any column names in later cells if they don't match what you actually see.

In [2]:
# --- Setup: install, auth, config ---
!pip -q install duckdb

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")   # Colab Secrets panel — never paste the token in a cell

REPO = "hf://datasets/FlyRank/internship-warehouse"

MONTH = "2026-03"              # mid-panel month — iterate here
SEALED_TEST_MONTH = "2026-06"  # the `_sample` file's month — mechanics only, never label logic

con = duckdb.connect()

# Documented pattern from the dataset card itself (works in Colab, no full download needed).
# The token only ever exists as a runtime string inside this SQL call — it is never printed
# or stored as a literal anywhere in this notebook's saved source.
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

print("Setup OK. Iterating on month:", MONTH)


Setup OK. Iterating on month: 2026-03


In [3]:
# --- Register views over this month's partition + the two small dimension tables ---
# Real repo layout, confirmed via HfApi().list_repo_files():
#   dim_clients.parquet                                  <- single file at repo root
#   dim_content.parquet                                  <- single file at repo root
#   fact_content_daily_performance/month=YYYY-MM/data_0.parquet  <- one file per month folder
#   fact_content_daily_performance_sample.parquet        <- sealed test month, mechanics only

con.execute(f"""
    CREATE OR REPLACE VIEW daily AS
    SELECT * FROM read_parquet('{REPO}/fact_content_daily_performance/month={MONTH}/*.parquet')
""")
con.execute(f"""
    CREATE OR REPLACE VIEW dim_clients AS
    SELECT * FROM read_parquet('{REPO}/dim_clients.parquet')
""")
con.execute(f"""
    CREATE OR REPLACE VIEW dim_content AS
    SELECT * FROM read_parquet('{REPO}/dim_content.parquet')
""")

print(con.execute("DESCRIBE daily").df())


                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

**Schema check:** `dim_clients`'s real columns are confirmed from the dataset card:
`client_hash_id`, `is_active`, `has_gsc_access`, `has_ga4_access`, `access_profile`,
`client_created_date`, `client_updated_date`, `gsc_data_start`, `ga4_data_start`. The join
key is **`client_hash_id`** — an earlier draft of this notebook guessed `client_id`, which
is wrong; fixed throughout below.

`daily`'s exact columns are **not yet confirmed** — only `gsc_avg_position` is verified by
`SKILL.md`. The `DESCRIBE daily` output above is the source of truth: if `content_hash_id`,
`gsc_clicks`, `gsc_impressions`, `gsc_ctr`, `ga4_engaged_sessions`, or `ga4_data_available`
don't match what's actually there, fix the queries below before trusting any of their
output — don't guess past this point.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. **One row = one (client, content, day) observation.** Each row of `fact_content_daily_performance`
   is one pseudonymized client's one piece of content on one `report_date`, with that day's GSC
   (and where available, GA4) metrics.
2. **Table(s):** `fact_content_daily_performance` (the `month=2026-03` partition) is primary,
   joined to `dim_content` for content-level context and `dim_clients` to check each client's
   `gsc_data_start` / `ga4_data_start` before trusting a window for that client.
3. **Time window:** `report_date` between `2026-03-01` and `2026-03-31` inclusive — a mid-panel
   month. Deliberately **not** the `_sample` table (that's June 2026, the sealed final month —
   mechanics-testing only, per the panel warning) and not the panel's first month (immature
   per-client history).
4. **Predict / rank (proxy):** within this same month, rank (client, content) pairs by whether
   their **second-half** average daily clicks (days 16–31) are lower than their **first-half**
   average (days 1–15) — a same-month proxy for "declining," standing in for the real
   forward-looking label that would need next month's data (out of scope for this contract).
5. **Deliberately excluded:** `client_hash_id` and `content_hash_id` as model features — they're
   pseudonymous join/grouping keys only, per the data skill's gotcha, never signal. Also
   excluded: any row's GA4 fields when `ga4_data_available` is not `TRUE` — those are
   zero-filled placeholders, not real "zero engagement," and get filtered rather than imputed.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Field | Why |
|---|---|---|
| Feature | `gsc_clicks` (H1 avg) | Observed GSC log, knowable at the decision moment |
| Feature | `gsc_impressions` (H1 avg) | Same — logged daily, no future information |
| Feature | `gsc_avg_position` (H1 avg) | Reflects ranking already measured, not a future outcome |
| Feature | `gsc_ctr` (H1 avg) | Derived same-day from clicks/impressions already observed |
| Feature | `ga4_engaged_sessions` (H1 avg, `ga4_data_available IS TRUE` only) | Real engagement signal where GA4 was actually live for that client |
| Label (proxy) | `declining` | 1 if H2 avg clicks < H1 avg clicks, else 0 — **only used to build the label, never as a feature** |
| Context | `client_hash_id`, `content_hash_id` | Join/group keys only — never features (pseudonyms) |
| Context | `report_date` | Used to build H1/H2 split, not a feature itself |
| Excluded | GA4 fields where `ga4_data_available` is not `TRUE` | Zero-filled placeholder, not a real zero — filtering, not imputing |
| Excluded | Any `*_h2` value as a feature | This is exactly the leakage trap in Section 3 — H2 data defines the label |


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Grain check — one row really is (client, content, day)

In [4]:
# Grain probe: should return ZERO rows. If it returns any, the grain claim in Section 1 is wrong.
grain_check = con.execute("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM daily
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Rows violating the stated grain (should be empty):")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the stated grain (should be empty):


,client_hash_id,content_hash_id,report_date,n


### 3b. Row count + date span for this slice

In [5]:
slice_summary = con.execute("""
    SELECT
        COUNT(*)                    AS n_rows,
        COUNT(DISTINCT client_hash_id)   AS n_clients,
        COUNT(DISTINCT content_hash_id)  AS n_content,
        MIN(report_date)            AS min_date,
        MAX(report_date)            AS max_date
    FROM daily
""").df()

slice_summary


,n_rows,n_clients,n_content,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


### 3c. Availability check — filtered with `IS TRUE`, not truthy-by-accident

In [6]:
availability = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(
            100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*),
            1
        ) AS pct_available
    FROM daily
""").df()

availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.2


In [8]:
schema = con.execute("DESCRIBE daily").df()
import pandas as pd
pd.set_option("display.max_rows", None)
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

### 3d. Five-feature frame (decision moment = end of March 15)

Every feature below is built **only from days 1–15** — nothing from the second half of the
month, which is where the label lives.

In [10]:
half = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_clicks END)      AS clicks_h1,
        AVG(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions END) AS impressions_h1,
        AVG(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_avg_position END) AS position_h1,
        AVG(CASE WHEN report_date < DATE '{MONTH}-16' AND gsc_impressions > 0
                 THEN gsc_clicks * 1.0 / gsc_impressions END)                    AS ctr_h1,
        AVG(CASE WHEN report_date < DATE '{MONTH}-16' AND ga4_data_available IS TRUE
                 THEN ga4_engaged_sessions END)                                  AS engaged_sessions_h1,
        AVG(CASE WHEN report_date >= DATE '{MONTH}-16' THEN gsc_clicks END)      AS clicks_h2
    FROM daily
    GROUP BY client_hash_id, content_hash_id
    HAVING clicks_h1 IS NOT NULL AND clicks_h2 IS NOT NULL
""").df()

half["declining"] = (half["clicks_h2"] < half["clicks_h1"]).astype(int)

feature_cols = [
    "clicks_h1", "impressions_h1", "position_h1", "ctr_h1", "engaged_sessions_h1",
]

print(half[feature_cols + ["declining"]].describe())
half.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

           clicks_h1  impressions_h1    position_h1         ctr_h1  \
count  319758.000000   319758.000000  151980.000000  151980.000000   
mean        0.080496       26.625425      15.653035       0.004698   
std         0.614620      126.163034      17.658603       0.038514   
min         0.000000        0.000000       0.000000       0.000000   
25%         0.000000        0.000000       4.817037       0.000000   
50%         0.000000        0.000000       8.285743       0.000000   
75%         0.000000        6.000000      19.753293       0.001966   
max       159.666667    10771.666667     310.000000       1.000000   

       engaged_sessions_h1      declining  
count         50746.000000  319758.000000  
mean              0.046698       0.114021  
std               0.157851       0.317837  
min               0.000000       0.000000  
25%               0.000000       0.000000  
50%               0.000000       0.000000  
75%               0.000000       0.000000  
max              

,client_hash_id,content_hash_id,clicks_h1,impressions_h1,position_h1,ctr_h1,engaged_sessions_h1,clicks_h2,declining
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,0.400000,278.200000,6.327311,0.001631,NaN,0.0625,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0.000000,16.333333,3.906852,0.000000,NaN,0.0000,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,0.200000,247.000000,6.473735,0.000771,NaN,0.1875,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,0.533333,162.666667,7.259861,0.004762,NaN,0.3125,1
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,0.000000,0.933333,9.000000,0.000000,NaN,0.0000,0


**Five features, "knowable at the decision moment because…":**

1. `clicks_h1` — daily organic clicks already logged by GSC for days 1–15; nothing about
   days 16–31 is needed to compute it.
2. `impressions_h1` — same reasoning; a same-day GSC log entry, not a forecast.
3. `position_h1` — the ranking position GSC already measured during the first half; it
   describes the past, not the outcome we're trying to predict.
4. `ctr_h1` — derived same-day from `clicks_h1` / `impressions_h1`, both already observed.
5. `engaged_sessions_h1` — real GA4 engagement, but only where `ga4_data_available IS TRUE`
   for that client during the first half; never backfilled with a zero that could be
   mistaken for "no engagement."


### 3e. The leakage trap — add ONE label-derived column on purpose

`clicks_h2` is the *raw material the label is built from*. It should never be a feature —
but let's prove why, the way notebook 02 did, instead of just asserting it.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X_honest = half[feature_cols].fillna(0)
y = half["declining"]

honest_scores = cross_val_score(
    LogisticRegression(max_iter=1000), X_honest, y, cv=5, scoring="accuracy"
)
print("Honest quick score (5 features, H1 only):", honest_scores.mean().round(3))


Honest quick score (5 features, H1 only): 0.898


In [12]:
# --- The trap: add ONE column derived straight from the label's own window ---
X_leaky = X_honest.copy()
X_leaky["clicks_h2"] = half["clicks_h2"].fillna(0)   # <-- this IS the label's raw material

leaky_scores = cross_val_score(
    LogisticRegression(max_iter=1000), X_leaky, y, cv=5, scoring="accuracy"
)
print("Leaky quick score (6 features, includes clicks_h2):", leaky_scores.mean().round(3))
print("Jump from honest -> leaky:", round(leaky_scores.mean() - honest_scores.mean(), 3))


Leaky quick score (6 features, includes clicks_h2): 0.98
Jump from honest -> leaky: 0.082


In [13]:
# --- Delete the leak column. Keep the honest number. ---
del X_leaky  # not used again — clicks_h2 is a context/label-building column only, never a feature

print("Final, honest quick score to carry into the modeling weeks:", honest_scores.mean().round(3))


Final, honest quick score to carry into the modeling weeks: 0.898


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** `dim_clients` contains stub rows for some `client_hash_id` values —
`access_profile = 'source_only_missing_client_dimension'`, with `is_active`, `has_gsc_access`,
`has_ga4_access` all null and no real `gsc_data_start`. These are client hashes that show up
in the fact/source data but were never fully onboarded into the client dimension. A plain
`daily JOIN dim_clients` will either silently drop those rows (inner join) or carry them
through with null access flags that look like "no access" when really it just means "no
dimension record" (left join) — two different silent failure modes, neither of which this
data can tell you apart from a real access-status change. This data also can't distinguish,
for those stub rows, whether the client ever had GSC/GA4 access at all — you'd need to go
back to the source system, not this warehouse release, to resolve it. Any lane that joins
to `dim_clients` for panel-maturity filtering needs to explicitly decide (and state) how it
handles these stubs rather than let a join silently pick for it.

In [14]:
# Prove the limitation above with a query, not just an assertion.
client_dim_health = con.execute("""
    SELECT
        access_profile,
        COUNT(*) AS n_clients,
        SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS null_gsc_start
    FROM dim_clients
    GROUP BY access_profile
    ORDER BY n_clients DESC
""").df()

client_dim_health


,access_profile,n_clients,null_gsc_start
0,gsc_and_ga4,53,3.0
1,no_search_or_analytics_access,26,23.0
2,gsc_only,14,4.0
3,source_only_missing_client_dimension,10,6.0
4,ga4_only,1,1.0


In [15]:
# How many of THIS month's fact rows belong to a client that only exists as a stub?
stub_row_impact = con.execute("""
    SELECT
        COUNT(*) AS total_rows_this_month,
        SUM(CASE WHEN c.access_profile = 'source_only_missing_client_dimension'
                 THEN 1 ELSE 0 END) AS rows_from_stub_clients
    FROM daily d
    LEFT JOIN dim_clients c ON d.client_hash_id = c.client_hash_id
""").df()

stub_row_impact


,total_rows_this_month,rows_from_stub_clients
0,9841378,3751.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
